# RetailPulse — the data, and what drifted

This notebook looks at the input side: what the panel actually contains, why MAPE cannot be
the headline metric, and what the year-over-year drift report says. It reads committed
aggregates only, so it runs on a fresh clone in seconds.

The drift section is the one to read skeptically. A year-over-year comparison over two
years of generated data can only ever describe the generator, so the honest conclusion is
that the PSI pipeline is demonstrably wired up and demonstrably reproducible, not that a
real retail panel would show this much or this little movement. The report says the same.

In [ ]:
import json
from pathlib import Path

import pandas as pd

PROCESSED = Path("data/processed")
DRIFT = Path("reports/drift")
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

for required in (PROCESSED, DRIFT):
    if not required.exists():
        raise SystemExit(f"{required} is missing.")

print("processed artefacts:", len(list(PROCESSED.glob("*.csv"))))
print("drift artefacts    :", sorted(p.name for p in DRIFT.iterdir()))

## The panel: shape, and the zeros that decide the metric

In [ ]:
panel = pd.read_csv(PROCESSED / "store_product_matrix.csv", index_col=0)
# Wide form: one row per store, one column per product, values are units. That is why the
# zero share below is a cell count rather than a row count -- 150 columns of 50 stores.
print(f"shape: {panel.shape[0]} stores x {panel.shape[1]} products")
print(f"total units: {panel.to_numpy().sum():,.0f}")
panel.iloc[:5, :8]


In [ ]:
values = panel.to_numpy()
zero_cells = int((values == 0).sum())
total_cells = values.size
print(f"zero cells: {zero_cells:,} of {total_cells:,} ({zero_cells/total_cells:.1%})")
print()
note = (
    "This is the fact that decides how F-03 is judged. A metric that divides by the\n"
    "actual is undefined on 77.9% of cells, so MAPE over the panel is not a number, it\n"
    "is an average over the 22% where one happens to exist. WAPE scales error by demand\n"
    "instead, so a zero week contributes zero error rather than dividing by zero."
)
print(note)


## Revenue distribution

In [ ]:
dist = pd.read_csv(PROCESSED / "demand_distribution.csv")
print("columns:", list(dist.columns))
dist.head(10)

In [ ]:
dist = pd.read_csv(PROCESSED / "demand_distribution.csv")
print(dist.head().to_string(index=False))
print("...")
print(dist.tail(3).to_string(index=False))


## Seasonality: where the peaks are

In [ ]:
peaks = pd.read_csv(PROCESSED / "category_peaks.csv")
print("columns:", list(peaks.columns))
peaks.head(12)

In [ ]:
monthly = pd.read_csv(PROCESSED / "category_monthly.csv")
print("shape:", monthly.shape)
monthly.head()

## F-07 — year-over-year drift

In [ ]:
drift = pd.read_csv(DRIFT / "drift_summary.csv")
print("column-level drift, reference year vs current year:")
print(drift[["column", "kind", "psi", "reference_mean", "current_mean", "verdict"]].to_string(index=False))


In [ ]:
category_drift = pd.read_csv(DRIFT / "category_mix_drift.csv")
print(category_drift.head(12).to_string(index=False))

In [ ]:
columns = ["product_category", "share_reference", "share_current", "share_shift", "psi"]
print("largest upward movers:")
print(category_drift.nlargest(3, "share_shift")[columns].to_string(index=False))
print()
print("largest downward movers:")
print(category_drift.nsmallest(3, "share_shift")[columns].to_string(index=False))


In [ ]:
report = DRIFT / "evidently_drift.html"
print(f"HTML report committed: {report.exists()}  ({report.stat().st_size/1024:.0f} KB)" if report.exists() else "HTML report MISSING")
print()
print("The HTML artefact is committed deliberately. A drift report that has to be")
print("regenerated to be looked at is not evidence, and tests/test_drift.py fails if")
print("the suite rewrites it.")

## What this section does and does not establish

In [ ]:
print("establishes:")
print("  - the panel shape and the zero-demand share, from committed aggregates")
print("  - why WAPE is the defensible headline metric for a sparse panel")
print("  - the drift pipeline runs, writes all three artefacts, and is reproducible")
print()
print("does not establish:")
print("  - that a real retail panel would drift this little; both years are generated")
print("  - that the drift threshold would fire correctly on unseen data; the threshold")
print("    is a convention, not something validated against labelled drift events")